# Gradio

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

Gradio turns a Python function into a shareable web UI in a few lines. You describe the
inputs and outputs as components, hand it your `fn`, and call `.launch()` — Gradio builds
the HTML/JS front end, runs a server, and (optionally) gives you a public share link. It is
the fastest path from "I have a model/function" to "someone non-technical can try it."

## 1. What & Why

**What it is.** Gradio is a Python library for building small web apps around functions —
demos, internal tools, model playgrounds. You never touch HTML, CSS, or JavaScript: you
compose **components** (`Textbox`, `Image`, `Audio`, `Slider`, `Chatbot`, …) and wire them
to a Python callable. Gradio renders the UI, marshals data between browser and Python,
and serves it over a built-in FastAPI/uvicorn server.

**The problem it solves.** Showing off a model usually means either a Jupyter cell only you
can run, or a from-scratch Flask/React app nobody has time to build. Gradio sits in the
middle: a real interactive UI, but defined inline in Python. Concretely it gives you:

- **Instant UI from a signature** — `gr.Interface(fn, inputs, outputs)` infers a usable
  layout; no front-end code.
- **Rich ML components** — image/audio/video/file widgets, a `Chatbot`, dataframes, plots —
  with the pre/post-processing (decode an upload, encode a return value) handled for you.
- **Free public sharing** — `launch(share=True)` tunnels a temporary `*.gradio.live` URL so
  you can send a colleague a link without deploying anything.
- **A REST/queue API for free** — every demo is also callable programmatically via
  `gradio_client`, and `queue()` handles concurrency and progress.
- **Hugging Face Spaces** — push a `app.py` with a Gradio demo and it hosts for free.

**Reach for it when** you want a quick demo, an internal labeling/eval tool, or a model
playground. **Skip it when** you need a polished multi-page product with custom design,
auth, and routing — that's a real web framework's job (or Gradio's heavier cousin for data
apps, Streamlit, only marginally so).

## 2. Mental Model

Think of Gradio as **`fn(inputs) -> outputs`, with a web form bolted onto each side.**

```
  browser widget        Gradio component         your Python
  ┌────────────┐  raw   ┌────────────────┐ python ┌──────────┐
  │  Textbox   │ ─────▶ │  preprocess()  │ ─────▶ │   fn()   │
  └────────────┘ value  └────────────────┘ value  └────┬─────┘
                                                        │ return
  ┌────────────┐ render ┌────────────────┐  value  ┌───▼──────┐
  │  Label     │ ◀───── │  postprocess() │ ◀────── │  output  │
  └────────────┘        └────────────────┘         └──────────┘
```

Every component knows how to **preprocess** a browser value into something Python-friendly
(an uploaded image → a NumPy array) and **postprocess** your return value back into
something the browser renders (a dict of class→score → a bar `Label`). Your function only
ever sees and returns plain Python.

Two ways to assemble those components:

- **`Interface`** — the high-level shortcut: "these inputs, this function, these outputs."
  Gradio picks the layout. Great for a single fn.
- **`Blocks`** — the low-level layout engine: you place components inside `Row`/`Column`,
  then explicitly bind events (`btn.click(fn, inputs, outputs)`). Use it for anything with
  multiple steps, shared state, or custom layout.

## 3. Key Concepts

| Term | What it is |
|------|------------|
| **Component** | A typed input/output widget (`Textbox`, `Slider`, `Image`, `Audio`, `Chatbot`, `Dataframe`, …). Each defines its own pre/post-processing. |
| **`Interface`** | High-level wrapper: `gr.Interface(fn, inputs, outputs)`. Builds a standard layout for one function. |
| **`Blocks`** | Low-level, imperative app builder. You lay out components and wire events yourself for full control. |
| **Event listener** | The wiring: `component.event(fn, inputs, outputs)` — e.g. `btn.click(...)`, `tb.change(...)`, `demo.load(...)`. |
| **`launch()`** | Starts the server. Key kwargs: `share=True` (public link), `server_name="0.0.0.0"`, `auth=(user, pw)`, `prevent_thread_lock=True` (don't block). |
| **`State`** | `gr.State()` holds per-session Python data (e.g. conversation history) across events without rendering anything. |
| **`queue()`** | Enables the request queue: concurrency limits, progress bars, and streaming. On by default in modern Gradio. |
| **`gr.ChatInterface`** | A purpose-built `Interface` for chatbots — handles history, streaming, and the message UI. |
| **`gradio_client`** | Python/JS client that calls any running demo as an API, so a demo doubles as a microservice. |
| **Spaces** | Hugging Face's free hosting; a repo with `app.py` + `requirements.txt` becomes a live Gradio app. |

## 4. Setup

Pure-Python install, CPU-friendly. The core library pulls in FastAPI/uvicorn and a few
data helpers; no GPU or model is required to run a demo.

```bash
%pip install gradio                 # the library + built-in server
# %pip install gradio_client        # to call a running demo as an API
```

The cells below **build and inspect** demos without starting a blocking server, so the
notebook runs top-to-bottom in CI. The one cell that actually serves a UI is gated behind
an environment variable — flip it on locally to see the real thing.

In [1]:
import os

# Keep CI quiet and offline: disable Gradio's version-check/analytics phone-home.
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")

import gradio as gr

print("gradio", gr.__version__)

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


gradio 6.19.0


## 5. Worked Examples

### Example 1 — `Interface`: a function becomes a UI

The classic three-argument form. We give Gradio a function, a list of input components, and
an output component; it figures out the layout. Instead of launching a server here, we
**inspect what Gradio built** and call the wrapped function directly so you can see the real
value the UI would render.

In [2]:
def greet(name, intensity):
    """Return a greeting, shouted `intensity` times."""
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=[
        gr.Textbox(label="name", value="Ada"),
        gr.Slider(minimum=1, maximum=5, step=1, value=3, label="intensity"),
    ],
    outputs=gr.Textbox(label="greeting"),
    title="Greeter",
    description="A function with a web form on each side.",
)

# What did Gradio assemble? (No server needed to look.)
print("built a    :", type(demo).__name__)
print("inputs     :", [c.__class__.__name__ for c in demo.input_components])
print("outputs    :", [c.__class__.__name__ for c in demo.output_components])
print("fn output  :", greet("Ada", 3))

built a    : Interface
inputs     : ['Textbox', 'Slider']
outputs    : ['Textbox']
fn output  : Hello, Ada!!!


Gradio inferred a two-field form on the left and a textbox on the right, all from the
components we listed. To actually serve it you'd call `demo.launch()` — that's gated in a
later cell so this notebook stays non-blocking.

### Example 2 — `Blocks`: custom layout and event wiring

`Blocks` is the imperative API: you place components and bind events yourself. Here a button
click runs a toy sentiment scorer whose result feeds a `gr.Label` (which postprocesses a
`{class: score}` dict into a bar chart). We confirm the event got wired and run the function
directly to see its output.

In [3]:
def sentiment(text):
    """Toy scorer: confidence split between positive/negative by keyword counts."""
    t = text.lower()
    pos, neg = t.count("good"), t.count("bad")
    total = pos + neg or 1  # avoid divide-by-zero on neutral text
    return {"positive": pos / total, "negative": neg / total}

with gr.Blocks(title="Sentiment") as blocks:
    gr.Markdown("## Mini sentiment demo")
    with gr.Row():
        inp = gr.Textbox(label="text", value="this is good good but a bit bad")
        out = gr.Label(label="scores")
    btn = gr.Button("Analyze", variant="primary")
    btn.click(fn=sentiment, inputs=inp, outputs=out)

print("event listeners wired:", len(blocks.fns))
print("sentiment('good good bad') ->", sentiment("good good bad"))

event listeners wired: 1
sentiment('good good bad') -> {'positive': 0.6666666666666666, 'negative': 0.3333333333333333}


One click event is registered, mapping `inp -> sentiment -> out`. Swap the toy function for
a real model call and the UI is unchanged — that separation of *layout* from *logic* is the
whole point of `Blocks`.

### Example 3 — actually serving it (gated)

Launching starts a real web server, which would block `nbconvert`, so we hide it behind an
env var and show the call shape. Run `GRADIO_RUN_SERVER=1` (locally, in a Jupyter kernel)
to bring up a live UI; `prevent_thread_lock=True` keeps the cell from hanging so you can
`demo.close()` afterward.

In [4]:
if os.getenv("GRADIO_RUN_SERVER"):
    # Non-blocking launch: returns immediately, server runs in a background thread.
    app, local_url, share_url = demo.launch(prevent_thread_lock=True)
    print("serving at:", local_url)        # e.g. http://127.0.0.1:7860
    # ... interact in the browser, then free the port:
    demo.close()
else:
    print("Set GRADIO_RUN_SERVER=1 to launch a live UI. Call shape:")
    print("    demo.launch()                      # blocking, opens http://127.0.0.1:7860")
    print("    demo.launch(share=True)            # + public *.gradio.live tunnel")
    print("    demo.launch(auth=('user', 'pw'))   # + basic auth")

Set GRADIO_RUN_SERVER=1 to launch a live UI. Call shape:
    demo.launch()                      # blocking, opens http://127.0.0.1:7860
    demo.launch(share=True)            # + public *.gradio.live tunnel
    demo.launch(auth=('user', 'pw'))   # + basic auth


## 6. Gotchas & Pitfalls

- **`launch()` blocks the script/cell.** In a plain `.py` script the process sits on the
  server. Use `prevent_thread_lock=True` to keep going, and `demo.close()` to release the
  port. Re-running a cell that launched often hits *"port 7860 in use"* — close the old
  demo first or pass `server_port=`.
- **`share=True` is a public tunnel.** Anyone with the `*.gradio.live` link can hit your
  function (and your machine) for the ~72h the link lives. Never expose unsafe functions;
  add `auth=` for anything sensitive.
- **Input/output count must match the function signature.** Three input components ⇒ `fn`
  takes three positional args; returning two values ⇒ two output components (or a tuple).
  Mismatches throw at call time, not at build time.
- **Components carry types — return the right shape.** `gr.Label` wants a `{label: score}`
  dict, `gr.Image` wants a path / NumPy array / PIL image, `gr.Dataframe` wants a list or
  DataFrame. Returning a bare string to an `Image` fails.
- **`gr.State` for per-user memory, not globals.** Module-level variables are shared across
  *all* sessions and will cross-contaminate users. Per-session data (chat history, a cart)
  belongs in `gr.State()`.
- **Blocking work freezes the UI; use the queue + `yield`.** Long jobs should stream partial
  results with a generator function (`yield`) and rely on the (default) queue; set
  `concurrency_limit` so one heavy request doesn't starve others.
- **Version churn.** Gradio's API moved a lot across 3.x → 4.x → 5.x → 6.x (component names,
  `chatbot` message format, event signatures). Pin a version, and check the migration notes
  before upgrading — examples from old tutorials may not run as-is.
- **It's a demo server, not a hardened backend.** No built-in rate limiting or robust auth.
  For production traffic, put it behind a real gateway or use a proper web framework.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses / when not to |
|--------|-----------|--------------------------|
| **Gradio** | Fastest fn → UI; rich ML components (image/audio/chat); free `share=` links; free API via `gradio_client`; first-class on HF Spaces. | Demo-grade server (weak auth/scaling); limited custom design/multi-page; API churn across majors. |
| **Streamlit** | Great for data dashboards and multi-widget exploratory apps; large ecosystem; script-reruns model is simple to reason about. | Rerun-the-whole-script model can be awkward for stateful flows; fewer turnkey ML I/O components; heavier for a single-function demo. |
| **Plain notebook (`ipywidgets`)** | Zero deploy; lives right next to your code; fine for solo exploration. | Only works inside Jupyter; not shareable as a URL to non-notebook users. |
| **FastAPI / Flask (+ React)** | Full control: routing, auth, design, scale — real product territory. | Weeks not minutes; you write the entire front end yourself. |
| **Panel / Dash** | Powerful for complex, callback-heavy analytical dashboards and plots. | Steeper learning curve; overkill for "wrap my model in a box." |

**Rule of thumb:** wrapping a model/function for people to *try* → **Gradio**. A data
exploration dashboard → **Streamlit** (or Dash/Panel if it's plot-heavy). A real
multi-page product with auth and custom UX → a **web framework**. Just you, in Jupyter →
**ipywidgets**.

## 8. Resources

- **Official docs** — https://www.gradio.app/docs
- **Quickstart / Getting started** — https://www.gradio.app/guides/quickstart
- **`Blocks` deep dive** (layout + events) — https://www.gradio.app/guides/blocks-and-event-listeners
- **Building a chatbot with `ChatInterface`** — https://www.gradio.app/guides/creating-a-chatbot-fast
- **`gradio_client` (use a demo as an API)** — https://www.gradio.app/guides/getting-started-with-the-python-client
- **Deploy free on Hugging Face Spaces** — https://huggingface.co/docs/hub/spaces-sdks-gradio
- **GitHub repo** — https://github.com/gradio-app/gradio